# 06 – Optimisation and performance

**Purpose:** analyse and improve the performance of the solution (report Section 6), comparing original and improved versions with timing routines, a profiler and FLOP estimates.

| # | What was optimised or analysed | Original → improved |
|---|---|---|
| 1 | Data pipeline runtime (profiled) | general timestamp parser → fixed-layout parser |
| 2 | Data pipeline memory | 2 M-row chunks → 500 k-row chunks |
| 3 | Learning algorithm | exact-split gradient boosting → histogram gradient boosting |
| 4 | Input resolution | number of histogram bins |
| 5 | Model size vs accuracy | number of trees |
| 6 | Repeatability and numerical stability | repeated fits, 1 vs 2 CPU threads |
| 7 | Arithmetic cost | FLOP and arithmetic-intensity estimates |

Sections 1–2 need the raw 1.6 GB file, so their measurements were saved to `reports/results/` by running the pipeline under a profiler; this notebook reads those files and re-runs a small parser benchmark that does not need the raw data. Accuracy, tuning, the threshold trade-off and the comparison with other methods are in notebooks 03–05.

In [1]:
import json, time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from occupancy import clean, evaluate, features
from occupancy.models import GradientBoostingForecaster, TUNED_SETTINGS
from occupancy.config import PROCESSED_DIR, REPO_ROOT

RESULTS = REPO_ROOT / "reports" / "results"
hourly = clean.load_hourly(PROCESSED_DIR / "occupancy_hourly.csv")
data = features.add_features(hourly)
split = evaluate.chronological_split(data)
train, val, test = split["train"], split["validation"], split["test"]
history = pd.concat([train, val])
mae = lambda d, p: evaluate.mae(d["mean_headcount"], p)

def timed(fn, repeats=3):
    best, out = float("inf"), None
    for _ in range(repeats):
        t = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t)
    return best, out

## 1. Data pipeline: profiling and the timestamp-parsing improvement
The cleaning step (9.1 M raw rows → 61 k hourly rows) is by far the slowest part of the workflow. Running it under Python's profiler (`cProfile`) showed that **over half of the time was spent converting timestamp text to dates**. The original code used pandas' general parser, which works out each value's format and UTC offset separately. Every raw timestamp has the same layout (`2024-05-14 13:02:10+10`), so the improved parser reads the first 19 characters with a fixed format (fast compiled code) and subtracts the whole-hour offset. It falls back to the general parser for any other layout.

Both runs below were made with the raw file already cached in memory, so they differ only in the code. The output table was byte-for-byte identical.

In [2]:
before = pd.read_csv(RESULTS / "stage6_pipeline_profile_before.csv")
after = pd.read_csv(RESULTS / "stage6_pipeline_profile.csv")
sb = json.load(open(RESULTS / "stage6_pipeline_summary_before.json"))
sa = json.load(open(RESULTS / "stage6_pipeline_summary.json"))
pipeline = pd.DataFrame({"original": sb, "improved": sa}).T[["total_seconds", "peak_memory_mb", "raw_rows", "hourly_rows"]]
pipeline["speed-up"] = pipeline.loc["original", "total_seconds"] / pipeline["total_seconds"]
print("Top functions by own time, ORIGINAL (profiler output):")
display(before.head(4).round(3))
print("Top functions by own time, IMPROVED:")
display(after.head(4).round(3))
pipeline.round(2)

Top functions by own time, ORIGINAL (profiler output):


,function,calls,own_seconds,cumulative_seconds,share_of_total
0,datetimes.py:objects_to_datetime64,5,53.389,53.390,0.616
1,c_parser_wrapper.py:read,6,15.006,15.516,0.173
2,merge.py:_factorize_keys,20,3.862,3.940,0.045
3,array.py:factorize,21,2.895,2.949,0.033


Top functions by own time, IMPROVED:


,function,calls,own_seconds,cumulative_seconds,share_of_total
0,c_parser_wrapper.py:read,6,14.459,14.959,0.385
1,merge.py:_factorize_keys,20,3.546,3.628,0.095
2,array.py:factorize,21,2.893,2.954,0.077
3,string_arrow.py:_from_sequence,89,2.171,2.181,0.058


,total_seconds,peak_memory_mb,raw_rows,hourly_rows,speed-up
original,86.63,1266.04,9091732.0,60988.0,1.00
improved,37.52,1284.33,9091732.0,60988.0,2.31


**Re-runnable check without the raw file:** the two parsers on 1 million timestamps in the raw layout (built from the hourly table).

In [3]:
rng = np.random.default_rng(0)
local = hourly["hour"].sample(1_000_000, replace=True, random_state=0).reset_index(drop=True)
local = local + pd.to_timedelta(rng.integers(0, 3600, len(local)), unit="s")
offset = (local.dt.tz_localize(None) - local.dt.tz_convert("UTC").dt.tz_localize(None)).dt.total_seconds() // 3600
strings = local.dt.strftime("%Y-%m-%d %H:%M:%S") + "+" + offset.astype(int).astype(str)
t_general, a = timed(lambda: clean.parse_utc_general(strings), repeats=1)
t_fast, b = timed(lambda: clean.parse_utc_fast(strings))
parse_bench = pd.DataFrame({"seconds": [t_general, t_fast]}, index=["general parser (original)", "fixed-layout parser (improved)"])
parse_bench["speed-up"] = t_general / parse_bench["seconds"]
print("Identical results:", a.equals(b), "| example string:", strings.iloc[0])
parse_bench.round(2)

Identical results: True | example string: 2024-09-12 20:51:02+10


,seconds,speed-up
general parser (original),6.20,1.00
fixed-layout parser (improved),0.66,9.33


## 2. Pipeline memory: chunk size
The raw file is read in chunks so it never has to fit in memory at once. Peak memory and runtime were measured for three chunk sizes (whole pipeline, raw file):

In [4]:
chunks = pd.read_json(RESULTS / "stage6_chunk_size.jsonl", lines=True).set_index("chunk_rows")
chunks.index = [f"{n:,} rows" + (" (whole file)" if n > 9_000_000 else "") for n in chunks.index]
chunks

,seconds,peak_memory_mb,hourly_rows
"500,000 rows",37.3,456,60988
"2,000,000 rows",37.3,1369,60988
"9,200,000 rows (whole file)",32.6,2939,60988


## 3. Algorithm: exact-split vs histogram gradient boosting
The *original* gradient-boosting algorithm (scikit-learn `GradientBoostingRegressor`) tests a split at every distinct value of every input. The *histogram* version used in this project first groups each input into ≤ 255 bins and tests only the bin edges. Both are run with the same settings (300 trees, learning rate 0.03, ≤ 31 leaves, ≥ 20 hours per leaf, squared-error loss, since the exact version has no Poisson option); the exact version cannot accept missing values, so they are filled with the training mean.

In [5]:
common = dict(learning_rate=0.03, max_leaf_nodes=31, min_samples_leaf=20)
X_cols = [f for f in features.FEATURES if f != "space"]
space_code = lambda d: d.assign(space_code=d["space"].astype("category").cat.codes)
exact = make_pipeline(SimpleImputer(strategy="mean"),
                      GradientBoostingRegressor(n_estimators=300, random_state=0, **common))
t_exact, _ = timed(lambda: exact.fit(space_code(train)[X_cols + ["space_code"]], train["mean_headcount"]), repeats=1)
pred_exact = np.clip(exact.predict(space_code(val)[X_cols + ["space_code"]]), 0, None)
hist_sq = GradientBoostingForecaster(max_iter=300, loss="squared_error", **common)
t_hist, _ = timed(lambda: hist_sq.fit(train))
t_tuned, tuned = timed(lambda: GradientBoostingForecaster(**TUNED_SETTINGS).fit(train))
algo = pd.DataFrame([
    {"version": "Exact splits (original algorithm), squared error", "fit_seconds": t_exact, "validation_mae": mae(val, pred_exact)},
    {"version": "Histogram (≤255 bins), squared error", "fit_seconds": t_hist, "validation_mae": mae(val, hist_sq.predict(val))},
    {"version": "Histogram + Poisson loss (selected)", "fit_seconds": t_tuned, "validation_mae": mae(val, tuned.predict(val))},
]).set_index("version")
algo["speed-up vs exact"] = t_exact / algo["fit_seconds"]
algo.to_csv(RESULTS / "stage6_exact_vs_histogram.csv")
algo.round(3)

,fit_seconds,validation_mae,speed-up vs exact
version,,,
"Exact splits (original algorithm), squared error",11.001,0.603,1.000
"Histogram (≤255 bins), squared error",1.208,0.609,9.106
Histogram + Poisson loss (selected),1.389,0.595,7.922


## 4. Input resolution: number of histogram bins
Fewer bins = coarser inputs and fewer candidate splits. How coarse can the inputs be before accuracy suffers?

In [6]:
rows = []
for bins in [8, 16, 32, 64, 128, 255]:   # at least 5 needed: one per space
    m = GradientBoostingForecaster(**TUNED_SETTINGS)
    m.params["max_bins"] = bins           # pass straight through to scikit-learn
    secs, m = timed(lambda: m.fit(train))
    rows.append({"max_bins": bins, "fit_seconds": secs, "validation_mae": mae(val, m.predict(val))})
bins = pd.DataFrame(rows).set_index("max_bins")
bins.to_csv(RESULTS / "stage6_bins.csv")
bins.round(3)

,fit_seconds,validation_mae
max_bins,,
8,1.152,0.614
16,1.087,0.602
32,1.074,0.600
64,1.181,0.597
128,1.333,0.599
255,1.426,0.595


## 5. Model size: number of trees
Accuracy on 2025 after each number of trees (one model with 1,000 trees, scored after every stage), to check whether 300 trees is enough, too few or wasteful. Training time grows in proportion to the number of trees.

In [7]:
big = GradientBoostingForecaster(**{**TUNED_SETTINGS, "max_iter": 1000}).fit(train)
Xv = big._matrix(val)
staged = {i + 1: mae(val, np.clip(p, 0, None)) for i, p in enumerate(big.model_.staged_predict(Xv))}
trees = pd.Series(staged, name="validation_mae")
pick = [25, 50, 100, 200, 300, 400, 600, 800, 1000]
trees_table = trees.loc[pick].to_frame()
trees_table.to_csv(RESULTS / "stage6_trees.csv")
print(f"Lowest validation MAE {trees.min():.4f} at {trees.idxmin()} trees; at 300 trees: {trees[300]:.4f}")
trees_table.round(4).T

Lowest validation MAE 0.5944 at 343 trees; at 300 trees: 0.5946


,25,50,100,200,300,400,600,800,1000
validation_mae,0.7398,0.6592,0.6114,0.5971,0.5946,0.5945,0.5962,0.5972,0.5976


## 6. Repeatability and numerical stability
* **Repeatability:** fit the selected model twice on the same data.
* **Threads:** fit with 1 CPU thread and with 2. Parallel sums can be added in a different order, which in floating-point arithmetic can change results slightly.
* **Stability:** the Poisson model works on log(headcount) and converts back with exp(), so forecasts are always finite and positive.

In [8]:
m1 = GradientBoostingForecaster(**TUNED_SETTINGS).fit(train).predict(test)
m2 = GradientBoostingForecaster(**TUNED_SETTINGS).fit(train).predict(test)
with threadpool_limits(limits=1):
    t1, m_one = timed(lambda: GradientBoostingForecaster(**TUNED_SETTINGS).fit(train), repeats=1)
with threadpool_limits(limits=2):
    t2, m_two = timed(lambda: GradientBoostingForecaster(**TUNED_SETTINGS).fit(train), repeats=1)
p1, p2 = m_one.predict(test), m_two.predict(test)
pd.Series({
    "two fits identical": bool(np.array_equal(m1, m2)),
    "max difference, 1 vs 2 threads (people)": float(np.max(np.abs(p1 - p2))),
    "same HVAC decision in every hour, 1 vs 2 threads": bool(((p1 >= 1) == (p2 >= 1)).all()),
    "all forecasts finite and ≥ 0": bool(np.isfinite(m1).all() and (m1 >= 0).all()),
    "fit seconds, 1 thread": round(t1, 2), "fit seconds, 2 threads": round(t2, 2),
})

two fits identical                                  True
max difference, 1 vs 2 threads (people)              0.0
same HVAC decision in every hour, 1 vs 2 threads    True
all forecasts finite and ≥ 0                        True
fit seconds, 1 thread                               1.48
fit seconds, 2 threads                               1.6
dtype: object

## 7. FLOP and arithmetic-intensity estimates
Order-of-magnitude estimates (not exact counts), with every assumption stated. *n* = training hours, *f* = inputs, *T* = trees, *L* = leaves per tree, *B* = bins.

**Histogram gradient boosting, per tree:**
* gradients and hessians of the Poisson loss: one `exp` and ~4 arithmetic operations per hour → ≈ 5*n* FLOPs;
* building histograms: each hour adds its gradient and hessian into one bin per input (2 additions × *f*), at each tree level it passes through. With *L* = 31 leaves the trees are about 5 levels deep, and scikit-learn builds only the smaller child's histogram and gets the other by subtraction, so on average each hour is passed through ≈ log₂*L* / 2 + 1 ≈ 3.5 times → ≈ 2 · *f* · 3.5 · *n* FLOPs;
* finding splits: scanning *B* bins × *f* inputs × ~10 operations for each of the ~*L* nodes → ≈ 10 · *B* · *f* · *L* FLOPs;
* updating predictions: ≈ 2*n*.

**Arithmetic intensity** (FLOPs per byte moved from memory) of the dominant histogram step: each addition reads a 1-byte bin index and an 8-byte gradient or hessian value → about 2 FLOPs per ~17 bytes ≈ 0.1 FLOP/byte. Values this low mean the step is limited by memory access, not by arithmetic speed.

In [9]:
n, f, T, L, B = len(history), len(features.FEATURES), 300, 31, 255
passes = np.log2(L) / 2 + 1
per_tree = {"gradients (Poisson)": 5 * n, "histograms": 2 * f * passes * n,
            "split finding": 10 * B * f * L, "prediction update": 2 * n}
flops_train = T * sum(per_tree.values())
fit_s, final = timed(lambda: GradientBoostingForecaster(**TUNED_SETTINGS).fit(history))

day = test[test["date"] == "2026-03-10"]
depth = np.log2(L) + 1
flops_predict_hour = T * (depth * 1 + 1)            # one comparison per level + one addition per tree
pred_s, _ = timed(lambda: final.predict(day), repeats=20)

PEAK_GFLOPS = 2 * 2.8 * 16    # 2 cores × 2.8 GHz × 16 double-precision FLOPs/cycle (one AVX-512 FMA unit): rough
flop_table = pd.DataFrame({
    "estimated FLOPs": [flops_train, flops_predict_hour * len(day)],
    "measured seconds": [fit_s, pred_s],
}, index=[f"Train ({n:,} hours, {T} trees)", f"Forecast one day ({len(day)} space-hours)"])
flop_table["achieved GFLOP/s"] = flop_table["estimated FLOPs"] / flop_table["measured seconds"] / 1e9
flop_table["% of rough CPU peak"] = 100 * flop_table["achieved GFLOP/s"] / PEAK_GFLOPS
flop_table.to_csv(RESULTS / "stage6_flops.csv")
print({k: f"{v * T / 1e6:.0f} MFLOP" for k, v in per_tree.items()})
print(f"Rough CPU peak assumed: {PEAK_GFLOPS:.0f} GFLOP/s")
flop_table.style.format({"estimated FLOPs": "{:.2e}", "measured seconds": "{:.4f}",
                         "achieved GFLOP/s": "{:.3f}", "% of rough CPU peak": "{:.2f}"})

{'gradients (Poisson)': '81 MFLOP', 'histograms': '1011 MFLOP', 'split finding': '213 MFLOP', 'prediction update': '32 MFLOP'}
Rough CPU peak assumed: 90 GFLOP/s


,estimated FLOPs,measured seconds,achieved GFLOP/s,% of rough CPU peak
"Train (53,828 hours, 300 trees)",1.34e+09,1.8846,0.710,0.79
Forecast one day (79 space-hours),1.65e+05,0.0104,0.016,0.02


For comparison, the **exact-split** algorithm sorts each input's values and tests every distinct value as a split point at every node, roughly *n* · *f* · log₂*n* operations per tree level instead of *n* · *f*, which is consistent with it being many times slower in Section 3. **Linear regression** solves one least-squares problem: about *n* · *p*² FLOPs for *p* ≈ 45 encoded inputs ≈ 10⁸ FLOPs, done once.

## 8. Observations
* **Profiling found the real bottleneck, and fixing it more than halved the pipeline runtime.** In the original pipeline, 62 % of the time was spent parsing timestamp text. The fixed-layout parser is about 8–9× faster at that step, cutting the whole cleaning run from 87 s to 38 s (2.3×) with byte-for-byte identical output. Reading the CSV file is now the largest cost (39 %).
* **Memory was cut by two-thirds at no cost in time** by reading 500,000 rows at a time instead of 2 million (peak 0.46 GB instead of 1.4 GB). Reading the whole file at once is about 13 % faster but needs 2.9 GB, which would fail on many laptops.
* **The histogram algorithm is about 9× faster than the original exact-split algorithm** (1.2 s vs 11.0 s; 9–10× across runs) for practically the same accuracy (MAE 0.609 vs 0.603 with the same squared-error loss). With the Poisson loss it is both faster and more accurate (0.595).
* **Input resolution:** as few as 32 bins gives MAE 0.600, and 255 bins gives 0.595. Training time hardly changes with the number of bins, so the default 255 is kept.
* **300 trees is close to the best.** Validation MAE falls steeply up to about 200 trees, is lowest at 343 trees (0.5944, against 0.5946 at 300), and slowly rises after that as the model starts to overfit. More trees would add training time for no gain.
* **Repeatable and numerically stable.** Two fits give identical forecasts. Using 1 or 2 CPU threads gives identical results and nearly the same time. Every forecast is finite and non-negative.
* **FLOPs.** Training needs roughly 1.3 billion floating-point operations, mostly building histograms, and runs at about 0.7–0.8 GFLOP/s, under 1 % of the processor's rough peak. That fits the low arithmetic intensity of histogram building (about 0.1 FLOP per byte): the work is limited by moving data through memory, not by arithmetic, which is also why a second thread barely helps. Forecasting a day needs only about 0.2 MFLOP, so its 0.01 s is almost entirely Python and pandas overhead.
